In [4]:
!pip install transformers torch accelerate pandas scikit-learn --quiet

import pandas as pd
from transformers import pipeline
from sklearn.metrics import confusion_matrix, classification_report

In [5]:
from google.colab import files
uploaded = files.upload()  # upload your full ~2537-headline CSV

full_df = pd.read_csv(next(iter(uploaded)))
print(full_df.head())
print("\nColumns:", full_df.columns.tolist())
print("Total headlines:", len(full_df))

Saving master_headlines.csv to master_headlines.csv
                           source  \
0                    Airdrie News   
1                      Al Jazeera   
2              Allafrica Tanzania   
3              Allafrica Tanzania   
4  Ani (asian News International)   

                                            headline            published  \
0  Commonwealth Games: Canada claims first gold m...  2026-07-24 21:18:58   
1  Which matches and tournaments can football fan...  2026-07-24 23:25:15   
2  Uganda: AFCON 2027 - A Golden Opportunity for ...  2026-07-24 20:41:56   
3  Uganda: Here Comes the Mandatory Monthly Natio...  2026-07-24 20:41:51   
4  Merck Foundation marks 'World Assisted Reprodu...  2026-07-25 00:00:00   

                                                link  
0  https://www.airdriecityview.com/national-sport...  
1  https://www.aljazeera.com/sports/2026/7/24/wha...  
2    https://allafrica.com/stories/202607240682.html  
3    https://allafrica.com/stories/2026072

In [6]:
general_classifier = pipeline(
    "sentiment-analysis",
    model="cardiffnlp/twitter-roberta-base-sentiment-latest",
    top_k=None  # returns confidence for all 3 classes, not just the top one
)

def score_headline(text):
    scores = general_classifier(str(text), truncation=True)[0]
    top = max(scores, key=lambda x: x["score"])
    return top["label"].capitalize(), top["score"]

results = full_df["headline"].astype(str).apply(score_headline)
full_df["Model_Label"] = results.apply(lambda x: x[0])
full_df["Model_Confidence"] = results.apply(lambda x: x[1])

print(full_df["Model_Label"].value_counts())
print("\nAvg confidence by predicted label:")
print(full_df.groupby("Model_Label")["Model_Confidence"].mean())

config.json:   0%|          | 0.00/929 [00:00<?, ?B/s]

pytorch_model.bin: reconstructing file:   0%|          |  0.00B /  501MB            

pytorch_model.bin: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

[transformers] RobertaForSequenceClassification LOAD REPORT from: cardiffnlp/twitter-roberta-base-sentiment-latest
Key                         | Status     |  | 
----------------------------+------------+--+-
roberta.pooler.dense.bias   | UNEXPECTED |  | 
roberta.pooler.dense.weight | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


...

vocab.json:   0%|          | 0.00/899k [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/456k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/239 [00:00<?, ?B/s]

[transformers] You seem to be using the pipelines sequentially on GPU. In order to maximize efficiency please use a dataset


Model_Label
Neutral     1663
Negative     615
Positive     258
Name: count, dtype: int64

Avg confidence by predicted label:
Model_Label
Negative    0.675985
Neutral     0.757953
Positive    0.689655
Name: Model_Confidence, dtype: float64


In [7]:
uploaded_manual = files.upload()  # upload manual_labels_sample.csv (the fixed version)

manual_df = pd.read_csv(next(iter(uploaded_manual)))
manual_df.columns = ["source", "headline", "published", "link", "sentiment"]
manual_df["sentiment"] = manual_df["sentiment"].str.strip().str.lower()

eval_df = pd.merge(
    manual_df,
    full_df[["headline", "Model_Label", "Model_Confidence"]],
    on="headline",
    how="inner"
)
eval_df["Model_Label"] = eval_df["Model_Label"].str.lower()

print("Rows matched:", len(eval_df))

cm = confusion_matrix(eval_df["sentiment"], eval_df["Model_Label"],
                       labels=["positive", "neutral", "negative"])
cm_df = pd.DataFrame(cm,
    index=["Actual Positive", "Actual Neutral", "Actual Negative"],
    columns=["Pred Positive", "Pred Neutral", "Pred Negative"])
print(cm_df)

print()
print(classification_report(eval_df["sentiment"], eval_df["Model_Label"], digits=3))

Saving manual_labels_sample.csv to manual_labels_sample.csv
Rows matched: 287
                 Pred Positive  Pred Neutral  Pred Negative
Actual Positive             32            89             16
Actual Neutral               6            72             15
Actual Negative              0            24             33

              precision    recall  f1-score   support

    negative      0.516     0.579     0.545        57
     neutral      0.389     0.774     0.518        93
    positive      0.842     0.234     0.366       137

    accuracy                          0.477       287
   macro avg      0.582     0.529     0.476       287
weighted avg      0.631     0.477     0.451       287



In [8]:
# Split: cardiffnlp-confident positives vs everything else
auto_positive_df = full_df[full_df["Model_Label"] == "Positive"].copy()
remaining_df = full_df[full_df["Model_Label"] != "Positive"].copy()

print("Auto-accepted as Positive:", len(auto_positive_df))
print("Remaining (needs second opinion):", len(remaining_df))

# Second model: siebert (binary, but good at catching negatives)
secondary_classifier = pipeline(
    "sentiment-analysis",
    model="siebert/sentiment-roberta-large-english",
    device=0,          # use GPU if available
    batch_size=32       # batched = much faster than row-by-row
)

headlines = remaining_df["headline"].astype(str).tolist()
secondary_results = secondary_classifier(headlines, truncation=True, batch_size=32)

remaining_df["Secondary_Label"] = [r["label"].capitalize() for r in secondary_results]
remaining_df["Secondary_Confidence"] = [r["score"] for r in secondary_results]

print(remaining_df["Secondary_Label"].value_counts())

Auto-accepted as Positive: 258
Remaining (needs second opinion): 2278


config.json:   0%|          | 0.00/687 [00:00<?, ?B/s]

pytorch_model.bin: reconstructing file:   0%|          |  0.00B / 1.42GB            

pytorch_model.bin: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/393 [00:00<?, ?it/s]

...

tokenizer_config.json:   0%|          | 0.00/256 [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/798k [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/456k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/150 [00:00<?, ?B/s]

Secondary_Label
Positive    1164
Negative    1114
Name: count, dtype: int64


In [9]:
import re

# Words that reliably signal negative real-world events on their own
NEGATIVE_KEYWORDS = [
    r"\bkill(ed|s|ing)?\b", r"\bdie[sd]?\b", r"\bdeath[s]?\b",
    r"\bdead\b", r"\bmurder(ed)?\b",
    r"\barrest(ed|s)?\b", r"\bcharged\b", r"\bconvict(ed|s|ion)?\b",
    r"\bjail(ed)?\b", r"\bprison\b",
    r"\bflood(s|ing)?\b", r"\bdrought\b", r"\bfamine\b", r"\bcrisis\b",
    r"\bcrash(es|ed)?\b", r"\bcollision\b", r"\baccident\b",
    r"\bprotest(s|ing|ers)?\b", r"\briot(s|ing)?\b", r"\bunrest\b",
    r"\bstrike[s]?\b(?!\s*out)",  # "strikes" = negative, but not "strikes out" (legal term)
    r"\battack(ed|s)?\b", r"\bviolence\b",
    r"\bcorrupt(ion)?\b", r"\bscandal\b", r"\bfraud\b", r"\bembezzle(ment|d)?\b",
    r"\bcollapse[sd]?\b", r"\bbankrupt(cy)?\b", r"\blayoff[s]?\b",
    r"\bban(ned|s)?\b", r"\bsuspend(ed|s)?\b",
    r"\brape[sd]?\b", r"\bassault(ed)?\b", r"\bkidnap(ped|ping)?\b",
]
pattern = re.compile("|".join(NEGATIVE_KEYWORDS), flags=re.IGNORECASE)

# "fire" alone is too ambiguous (metaphors like "Sifuna's fire", "fireman swagger")
# only count it negative when paired with actual disaster words nearby
FIRE_PATTERN = re.compile(
    r"\bfire\b.*\b(razes|guts|kills|died|dead|outbreak|dormitory|market|building)\b"
    r"|\b(razes|guts|dormitory|market)\b.*\bfire\b",
    flags=re.IGNORECASE
)

def matches_negative_keyword(headline):
    text = str(headline)
    return bool(pattern.search(text)) or bool(FIRE_PATTERN.search(text))

remaining_df["Keyword_Negative_Flag"] = remaining_df["headline"].apply(matches_negative_keyword)

print("Flagged as keyword-negative:", remaining_df["Keyword_Negative_Flag"].sum())
print("Of those, model agreement:")
print(remaining_df[remaining_df["Keyword_Negative_Flag"]]["Secondary_Label"].value_counts())

Flagged as keyword-negative: 353
Of those, model agreement:
Secondary_Label
Negative    274
Positive     79
Name: count, dtype: int64


In [10]:
eval_remaining = pd.merge(
    manual_df,
    remaining_df[["headline", "Model_Label", "Secondary_Label", "Secondary_Confidence"]],
    on="headline",
    how="inner"
)
eval_remaining["Secondary_Label"] = eval_remaining["Secondary_Label"].str.lower()

print("Rows matched (leftover bucket only):", len(eval_remaining))
print()

cm2 = confusion_matrix(
    eval_remaining["sentiment"], eval_remaining["Secondary_Label"],
    labels=["positive", "neutral", "negative"]
)
cm2_df = pd.DataFrame(cm2,
    index=["Actual Positive", "Actual Neutral", "Actual Negative"],
    columns=["Pred Positive", "Pred Neutral(never predicted)", "Pred Negative"])
print(cm2_df)

print()
print(classification_report(eval_remaining["sentiment"], eval_remaining["Secondary_Label"],
                             labels=["positive","neutral","negative"], digits=3, zero_division=0))

Rows matched (leftover bucket only): 249

                 Pred Positive  Pred Neutral(never predicted)  Pred Negative
Actual Positive             73                              0             32
Actual Neutral              41                              0             46
Actual Negative             11                              0             46

              precision    recall  f1-score   support

    positive      0.584     0.695     0.635       105
     neutral      0.000     0.000     0.000        87
    negative      0.371     0.807     0.508        57

    accuracy                          0.478       249
   macro avg      0.318     0.501     0.381       249
weighted avg      0.331     0.478     0.384       249



In [11]:
remaining_df["Auto_Negative_Candidate"] = (
    remaining_df["Keyword_Negative_Flag"] &
    ((remaining_df["Model_Label"] == "Negative") | (remaining_df["Secondary_Label"] == "Negative"))
)

print("Auto-negative candidates in full remaining set:", remaining_df["Auto_Negative_Candidate"].sum())

Auto-negative candidates in full remaining set: 290


In [12]:
# 1. Auto-accepted Positive
positive_export = auto_positive_df[["source", "headline", "published", "link"]].copy()
positive_export["sentiment"] = "positive"
positive_export["label_source"] = "auto_cardiffnlp_positive"

# 2. Auto-accepted Negative
auto_negative_df = remaining_df[remaining_df["Auto_Negative_Candidate"]].copy()
negative_export = auto_negative_df[["source", "headline", "published", "link"]].copy()
negative_export["sentiment"] = "negative"
negative_export["label_source"] = "auto_keyword_model_agreement"

# 3. Everything else -- still needs a human
needs_manual_df = remaining_df[~remaining_df["Auto_Negative_Candidate"]].copy()
manual_export = needs_manual_df[["source", "headline", "published", "link",
                                   "Model_Label", "Secondary_Label"]].copy()
manual_export["sentiment"] = ""  # you fill this in by hand

print(f"Positive (auto):      {len(positive_export)}")
print(f"Negative (auto):      {len(negative_export)}")
print(f"Needs manual review:  {len(manual_export)}")
print(f"Total:                {len(positive_export) + len(negative_export) + len(manual_export)}")

positive_export.to_csv("auto_labeled_positive.csv", index=False)
negative_export.to_csv("auto_labeled_negative.csv", index=False)
manual_export.to_csv("needs_manual_labeling.csv", index=False)

files.download("auto_labeled_positive.csv")
files.download("auto_labeled_negative.csv")
files.download("needs_manual_labeling.csv")

Positive (auto):      258
Negative (auto):      290
Needs manual review:  1988
Total:                2536


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

TRIAL


In [13]:
# transformers + torch give us the pretrained sentiment model
# (Colab usually has torch pre-installed; --quiet keeps the output clean)
!pip install transformers torch --quiet

In [14]:
from google.colab import files

print("Upload master_dataset.csv (your trusted/validated labels) and master_headlines.csv (full headline pool)")
uploaded = files.upload()  # a file picker will pop up - select both files

Upload master_dataset.csv (your trusted/validated labels) and master_headlines.csv (full headline pool)


Saving master_dataset.csv to master_dataset (1).csv


In [15]:
import pandas as pd

master_dataset = pd.read_csv("master_dataset.csv")      # trusted, 100% correct, never touched
master_headlines = pd.read_csv("master_headlines.csv")  # full pool of all headlines

# Build a matching key: prefer link, fall back to lowercased headline text
def make_key(row):
    link = str(row.get("link", "")).strip()
    if link and link.lower() != "nan":
        return f"link::{link}"
    return f"headline::{str(row.get('headline','')).strip().lower()}"

master_dataset["_key"] = master_dataset.apply(make_key, axis=1)
master_headlines["_key"] = master_headlines.apply(make_key, axis=1)

trusted_keys = set(master_dataset["_key"])

# Everything in the full pool that ISN'T already in your trusted set
to_label = master_headlines[~master_headlines["_key"].isin(trusted_keys)].drop_duplicates(subset="_key").copy()

print(f"Trusted (manual, untouched):  {len(master_dataset)}")
print(f"Still needs auto-labeling:    {len(to_label)}")

Trusted (manual, untouched):  762
Still needs auto-labeling:    1774


In [16]:
from transformers import pipeline

# cardiffnlp/twitter-roberta-base-sentiment-latest: 3-class sentiment model
# (negative / neutral / positive). This is the same model your earlier pilot used.
sentiment_model = pipeline(
    "sentiment-analysis",
    model="cardiffnlp/twitter-roberta-base-sentiment-latest",
    truncation=True
)

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

[transformers] RobertaForSequenceClassification LOAD REPORT from: cardiffnlp/twitter-roberta-base-sentiment-latest
Key                         | Status     |  | 
----------------------------+------------+--+-
roberta.pooler.dense.bias   | UNEXPECTED |  | 
roberta.pooler.dense.weight | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [17]:
# Run in batches so it's fast even for a few thousand headlines
headlines_list = to_label["headline"].astype(str).tolist()

predictions = sentiment_model(headlines_list, batch_size=32)

# predictions look like: [{'label': 'positive', 'score': 0.87}, ...]
to_label["model_label"] = [p["label"].lower() for p in predictions]
to_label["model_score"] = [p["score"] for p in predictions]

print(to_label["model_label"].value_counts())

model_label
neutral     1409
negative     365
Name: count, dtype: int64


In [18]:
# Positive/negative predictions: trust them (99.4% / 89.8% accurate per our validation)
confident = to_label[to_label["model_label"].isin(["positive", "negative"])].copy()
confident["sentiment"] = confident["model_label"]
confident["label_source"] = "auto_cardiffnlp"

# Neutral predictions: only ~27% accurate - do NOT auto-accept, send for manual review instead
needs_review = to_label[to_label["model_label"] == "neutral"].copy()

OUTPUT_COLUMNS = ["source", "headline", "published", "link", "sentiment", "label_source"]

# Final dataset = your untouched trusted rows + the confident auto-labeled rows
final_dataset = pd.concat(
    [master_dataset[OUTPUT_COLUMNS], confident[OUTPUT_COLUMNS]],
    ignore_index=True
)

print(f"Final dataset total rows:     {len(final_dataset)}")
print(f"  - trusted (manual):         {len(master_dataset)}")
print(f"  - auto-labeled (confident): {len(confident)}")
print(f"Sent to needs_review.csv:     {len(needs_review)}  (predicted neutral, low trust)")

final_dataset.to_csv("final_dataset.csv", index=False)
needs_review[["source", "headline", "published", "link", "model_label", "model_score"]].to_csv("needs_review.csv", index=False)

Final dataset total rows:     1127
  - trusted (manual):         762
  - auto-labeled (confident): 365
Sent to needs_review.csv:     1409  (predicted neutral, low trust)


---
# Milestone 2 Extension: Reducing the Remaining Manual Labeling Load

**Context:** After the TRIAL section above, we had `final_dataset.csv` (1,127 labeled headlines: 762 trusted + 365 confident cardiffnlp auto-labels) and `needs_review.csv` (1,409 headlines cardiffnlp couldn't confidently label — mostly predicted "neutral," where it's only ~27% accurate).

This section documents **two further attempts** to reduce that 1,409 manual workload, tested properly against ground truth rather than assumed:
1. VADER (lexicon-based sentiment) — **tested, did not help**
2. A domain-matched classifier trained on our own labeled data ("bootstrap" self-training) — **tested, worked, adopted**

Update the file paths in the first code cell below to match your local project structure before running.

## Setup: file paths

Adjust these to match your project structure.

In [5]:
from google.colab import files

print("Upload final_dataset.csv (1,127 labeled rows: 762 trusted + 365 cardiffnlp auto-labels)")
uploaded_final = files.upload()
FINAL_DATASET_PATH = list(uploaded_final.keys())[0]

print("Upload needs_review.csv (1,409 rows still needing labels)")
uploaded_review = files.upload()
NEEDS_REVIEW_PATH = list(uploaded_review.keys())[0]

print("Upload manual_labels_sample.csv (288-row ground truth, the fixed version)")
uploaded_manual = files.upload()
MANUAL_SAMPLE_PATH = list(uploaded_manual.keys())[0]

OUTPUT_AUTO_LABELED = "bootstrap_auto_labeled.csv"
OUTPUT_STILL_REVIEW = "still_needs_review.csv"

Upload final_dataset.csv (1,127 labeled rows: 762 trusted + 365 cardiffnlp auto-labels)


Saving final_dataset.csv to final_dataset.csv
Upload needs_review.csv (1,409 rows still needing labels)


Saving needs_review.csv to needs_review.csv
Upload manual_labels_sample.csv (288-row ground truth, the fixed version)


Saving manual_labels_sample.csv to manual_labels_sample (1).csv


In [6]:
import pandas as pd
import re
import nltk
nltk.download('stopwords', quiet=True)
nltk.download('wordnet', quiet=True)
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer

final_dataset = pd.read_csv(FINAL_DATASET_PATH).dropna(subset=['headline', 'sentiment'])
needs_review = pd.read_csv(NEEDS_REVIEW_PATH).dropna(subset=['headline'])
manual_sample = pd.read_csv(MANUAL_SAMPLE_PATH)
manual_sample['sentiment'] = manual_sample['sentiment'].str.strip().str.lower()

print(f"final_dataset.csv: {len(final_dataset)} rows")
print(final_dataset['sentiment'].value_counts())
print(f"\nneeds_review.csv: {len(needs_review)} rows")
print(f"\nmanual_labels_sample.csv (ground truth): {len(manual_sample)} rows")

final_dataset.csv: 1127 rows
sentiment
negative    692
positive    350
neutral      85
Name: count, dtype: int64

needs_review.csv: 1409 rows

manual_labels_sample.csv (ground truth): 288 rows


**Important imbalance to flag:** `final_dataset.csv` has only **85 neutral** examples out of 1,127 (vs. 350 positive, 692 negative). Any classifier trained on this will struggle to learn what "neutral" looks like — this shapes every decision below.

## Attempt A: VADER (lexicon-based sentiment)

VADER is a rule-based sentiment tool tuned for social media text (slang, punctuation, emoji). Tested directly against our 288-row manual ground truth, the same way cardiffnlp was validated earlier.

In [7]:
from google.colab import files

print("Upload manual_labels_sample.csv (the fixed version)")
uploaded_manual = files.upload()
MANUAL_SAMPLE_PATH = list(uploaded_manual.keys())[0]

import pandas as pd
manual_sample = pd.read_csv(MANUAL_SAMPLE_PATH)
manual_sample['sentiment'] = manual_sample['sentiment'].str.strip().str.lower()
print(f"Loaded {len(manual_sample)} rows")

Upload manual_labels_sample.csv (the fixed version)


Saving manual_labels_sample.csv to manual_labels_sample (2).csv
Loaded 288 rows


In [8]:
!pip install vaderSentiment --quiet
from vaderSentiment.vaderSentiment import SentimentIntensityAnalyzer

vader = SentimentIntensityAnalyzer()

manual_sample['vader_compound'] = manual_sample['headline'].apply(lambda t: vader.polarity_scores(str(t))['compound'])

def label_from_compound(compound):
    if compound > 0.05:
        return 'positive'
    elif compound < -0.05:
        return 'negative'
    return 'neutral'

manual_sample['vader_label'] = manual_sample['vader_compound'].apply(label_from_compound)

from sklearn.metrics import classification_report
print(classification_report(manual_sample['sentiment'], manual_sample['vader_label'], digits=3, zero_division=0))

              precision    recall  f1-score   support

    negative      0.404     0.667     0.503        57
     neutral      0.382     0.447     0.412        94
    positive      0.595     0.365     0.452       137

    accuracy                          0.451       288
   macro avg      0.460     0.493     0.456       288
weighted avg      0.488     0.451     0.449       288



**Result: VADER scored ~45% overall accuracy — worse than cardiffnlp's 47.7% ensemble attempt.** Even restricting to VADER's most confident predictions only, accuracy topped out around 58%.

**Conclusion: VADER does not help here.** This isn't a cardiffnlp-specific problem — it's that lexicon/social-media-tuned sentiment tools in general don't transfer well to Kenyan news headlines. This is itself a useful, generalizable finding for the report. **Attempt A abandoned; not used in the final pipeline.**

## Attempt B: Domain-matched classifier trained on our own labeled data ("bootstrap" self-training)

Instead of relying on an external pretrained model, train a simple classifier directly on our 1,127 already-labeled headlines, then use it to label the remaining pool. This is a standard semi-supervised technique (self-training).

### Step 1: Clean text (same method used throughout this notebook)

In [9]:
stop_words = set(stopwords.words('english'))
lemmatizer = WordNetLemmatizer()

def clean_text(text):
    text = re.sub('[^a-zA-Z]', ' ', str(text))
    text = text.lower().split()
    text = [lemmatizer.lemmatize(word) for word in text if word not in stop_words]
    return ' '.join(text)

final_dataset['clean_headline'] = final_dataset['headline'].apply(clean_text)
needs_review['clean_headline'] = needs_review['headline'].apply(clean_text)
final_dataset[['headline', 'clean_headline']].head()

,headline,clean_headline
0,KBL returns to court over alleged corruption i...,kbl return court alleged corruption sh bn arbi...
1,India's Gen Z protesters adapt to internet shu...,india gen z protester adapt internet shutdown
2,"Who is Karim Khan, the barrister at the center...",karim khan barrister center icc internal crisis
3,Body found dumped along Nairobi–Naivasha Highw...,body found dumped along nairobi naivasha highw...
4,"Two including Italian killed, two injured in R...",two including italian killed two injured rabai...


### Step 2: TF-IDF features + train/test split (to fairly evaluate which algorithm to use)

In [10]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.model_selection import train_test_split

tfidf = TfidfVectorizer(max_features=5000)
X = tfidf.fit_transform(final_dataset['clean_headline'])
y = final_dataset['sentiment'].values

Xtrain, Xtest, ytrain, ytest = train_test_split(X, y, test_size=0.20, random_state=42, stratify=y)
Xtrain.shape, Xtest.shape

((901, 4176), (226, 4176))

### Step 3: Compare 4 algorithms head-to-head

Testing Logistic Regression, Linear SVM, Naive Bayes, and Random Forest — with `class_weight='balanced'` where supported, to try to compensate for the 85-neutral imbalance. Comparing on **macro F1** (treats all 3 classes equally, so it won't hide neutral failing completely) and specifically checking whether each model can detect neutral **at all**.

In [11]:
from sklearn.linear_model import LogisticRegression
from sklearn.svm import LinearSVC
from sklearn.naive_bayes import MultinomialNB
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import f1_score

candidate_models = {
    'Logistic Regression (balanced)': LogisticRegression(max_iter=1000, class_weight='balanced'),
    'Linear SVM (balanced)': LinearSVC(max_iter=2000, class_weight='balanced'),
    'Naive Bayes': MultinomialNB(),
    'Random Forest (balanced)': RandomForestClassifier(n_estimators=200, class_weight='balanced', random_state=42),
}

comparison = []
for name, m in candidate_models.items():
    m.fit(Xtrain, ytrain)
    pred = m.predict(Xtest)
    f1m = f1_score(ytest, pred, average='macro')
    neutral_f1 = f1_score(ytest, pred, labels=['neutral'], average='macro', zero_division=0)
    comparison.append({'Model': name, 'Macro F1': round(f1m, 3), 'Neutral F1': round(neutral_f1, 3)})
    print(f"=== {name} ===")
    print(classification_report(ytest, pred, digits=3, zero_division=0))

comparison_df = pd.DataFrame(comparison).sort_values('Macro F1', ascending=False)
comparison_df

=== Logistic Regression (balanced) ===
              precision    recall  f1-score   support

    negative      0.794     0.885     0.837       139
     neutral      0.250     0.118     0.160        17
    positive      0.730     0.657     0.692        70

    accuracy                          0.757       226
   macro avg      0.591     0.553     0.563       226
weighted avg      0.733     0.757     0.741       226

=== Linear SVM (balanced) ===
              precision    recall  f1-score   support

    negative      0.765     0.935     0.841       139
     neutral      0.000     0.000     0.000        17
    positive      0.774     0.586     0.667        70

    accuracy                          0.757       226
   macro avg      0.513     0.507     0.503       226
weighted avg      0.710     0.757     0.724       226

=== Naive Bayes ===
              precision    recall  f1-score   support

    negative      0.694     0.978     0.812       139
     neutral      0.000     0.000     0.

,Model,Macro F1,Neutral F1
0,Logistic Regression (balanced),0.563,0.16
1,Linear SVM (balanced),0.503,0.00
2,Naive Bayes,0.431,0.00
3,Random Forest (balanced),0.426,0.00


**Result: Logistic Regression (balanced) wins on macro F1 (0.563), and is the ONLY model that detects any neutral cases at all** (weak — ~12% recall — but non-zero, vs. 0% for SVM/NB/Random Forest). This isn't a coincidence: `class_weight='balanced'` reweights probability estimates natively in Logistic Regression's math, while the other algorithms either ignore the reweighting internally or need much more data to benefit from it.

**Decision: use Logistic Regression (balanced) as the bootstrap labeler.**

### Step 4: Validate on held-out data BEFORE trusting it on real unlabeled data

Before using this model to label anything for real, check: when it's *confident* about a positive/negative prediction, how often is it actually right? This determines the confidence threshold to use.

In [12]:
import numpy as np

final_model_test = LogisticRegression(max_iter=1000, class_weight='balanced')
final_model_test.fit(Xtrain, ytrain)

probs = final_model_test.predict_proba(Xtest)
classes = final_model_test.classes_
preds = classes[probs.argmax(axis=1)]
conf = probs.max(axis=1)

for thresh in [0.5, 0.6, 0.7]:
    mask = (preds != 'neutral') & (conf >= thresh)
    if mask.sum() > 0:
        acc = (preds[mask] == ytest[mask]).mean()
        print(f"threshold {thresh}: n={mask.sum()}, accuracy on confident positive/negative predictions = {acc:.3f}")
    else:
        print(f"threshold {thresh}: no predictions meet this threshold")

threshold 0.5: n=115, accuracy on confident positive/negative predictions = 0.878
threshold 0.6: n=47, accuracy on confident positive/negative predictions = 0.894
threshold 0.7: n=12, accuracy on confident positive/negative predictions = 0.917


**Result: at threshold 0.5, confident positive/negative predictions are correct 87.8% of the time** — well above cardiffnlp's overall 47.7% and VADER's 45.1%. This is trustworthy enough to use for real auto-labeling. **Threshold chosen: 0.5.**

### Step 5: Train the final model on ALL of `final_dataset.csv`, apply to `needs_review.csv`

Now using the full 1,127 labeled rows (not just the 80% training split) to get the best possible final model, then predicting on the real remaining pool.

In [13]:
final_model = LogisticRegression(max_iter=1000, class_weight='balanced')
final_model.fit(X, y)   # train on ALL of final_dataset.csv, not just the split

X_review = tfidf.transform(needs_review['clean_headline'])
review_probs = final_model.predict_proba(X_review)
review_classes = final_model.classes_
review_preds = review_classes[review_probs.argmax(axis=1)]
review_conf = review_probs.max(axis=1)

needs_review['sentiment'] = review_preds
needs_review['confidence'] = review_conf
needs_review['label_source'] = 'auto_bootstrap_lr'

THRESHOLD = 0.5
trust_mask = (needs_review['sentiment'] != 'neutral') & (needs_review['confidence'] >= THRESHOLD)

bootstrap_auto_labeled = needs_review[trust_mask].copy()
still_needs_review = needs_review[~trust_mask].copy().sort_values('confidence', ascending=True)

print(f"Auto-labeled with confidence: {len(bootstrap_auto_labeled)}")
print(bootstrap_auto_labeled['sentiment'].value_counts())
print(f"\nStill needs manual review: {len(still_needs_review)}")

Auto-labeled with confidence: 535
sentiment
negative    284
positive    251
Name: count, dtype: int64

Still needs manual review: 874


### Step 6: Save outputs

`bootstrap_auto_labeled.csv` is ready to merge into `final_dataset.csv` via the existing `combine_labels.py` pattern. `still_needs_review.csv` is sorted so the shakiest (lowest-confidence) predictions are at the top — review those first.

In [14]:
bootstrap_auto_labeled.to_csv(OUTPUT_AUTO_LABELED, index=False)
still_needs_review.to_csv(OUTPUT_STILL_REVIEW, index=False)
print("Saved:", OUTPUT_AUTO_LABELED, "and", OUTPUT_STILL_REVIEW)

Saved: bootstrap_auto_labeled.csv and still_needs_review.csv


---
## Summary for the team

| Approach | Overall accuracy (vs. manual ground truth) | Verdict |
|---|---|---|
| cardiffnlp (Attempt 1 ensemble) | 47.7% | Used for 762-row trusted core + 365 confident auto-labels |
| VADER | ~45.1% | **Rejected** — no better than cardiffnlp, doesn't transfer to news headlines |
| Bootstrap classifier (Logistic Regression, trained on our own data) | 87.8% on confident predictions | **Adopted** — auto-labeled 535 more headlines |

**Manual labeling workload: 1,409 → 874 headlines remaining** (a real ~38% reduction, verified against held-out data rather than assumed).

**Known limitation to carry into the final report:** none of the approaches tried (cardiffnlp, VADER, or the bootstrap classifier) reliably detect the "neutral" class — this is a direct consequence of having only 85 neutral examples in the trusted training data. `still_needs_review.csv` is therefore neutral-heavy by design, and neutral headlines should be prioritized for manual labeling to eventually fix this imbalance for good.

**Next step:** merge `bootstrap_auto_labeled.csv` into `final_dataset.csv`, bringing the trusted labeled set to ~1,662 rows — this becomes the training data for Milestone 3.

In [15]:
from google.colab import files

files.download("final_dataset.csv")
files.download("needs_review.csv")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>